In [4]:
import numpy as np
from tqdm.notebook import tqdm
%run getRegularH.ipynb import getRegularH

In [5]:
# check node update function
def MP_BEC_CN(x):
    y = np.full_like(x, 2) # all messages erased
    known = x != 2 # determine positions that are known (either 0 or 1)

    # Case 1, no erasure
    if np.all(known): 
        y[:] = np.prod(x)
    # Case 2, one single erasures
    elif np.sum(~known) == 1:
        y[~known] = np.prod(x[known])

    return y

# variable node 
def MP_BEC_VN(x, yc):
    y = np.full_like(x, yc)
    known = x != 2

    if yc == 2: # channel is erased
        if np.sum(known) == 1: # all erased but one message
            y[~known] = x[known][0]
        elif np.sum(known) > 1: # at least two non-erased messages
            y[:] = x[known][0]

    return y

# LDPC decoder for the BEC, inner loop
# completely vectorized but still relatively slow
# based on a sparse matrix as input
def decode_LDPC_BEC(y, H, iterations):
    m, n = H.shape # get row and column indices

    # initialize variable to check node messages with channel output
    VtoC = np.tile(y, (m, 1)) * H
    # initialize check to variable node messages with erasures
    CtoV = np.full((m, n), 2) * H

    # main iterations 
    for _ in range(iterations):

        # iterate through all check nodes  
        for i in range(m):
            idx = H[i] == 1
            CtoV[i, idx] = MP_BEC_CN(VtoC[i, idx])

        # iterate through all variable nodes
        for j in range(n):
            idx = H[:, j] == 1
            VtoC[idx, j] = MP_BEC_VN(CtoV[idx, j], y[j])

        # binary decision
        x_dec = y.copy()
        for j in range(n):
            msgs = CtoV[H[:, j] == 1, j]
            if np.any(msgs != 2):
                x_dec[j] = np.min(msgs)

        if np.all(x_dec < 2):
            xh = (1 - x_dec) // 2
            if np.all((H @ xh) % 2 == 0):
                # all parity-checks fulfilled?
                return xh

    return np.array([])

def simulate_LDPC_BEC():
    # parameters of regular LDPC code
    dv = 3
    dc = 6

    # size of code
    n = 200

    # specify epsilon (erasure probability) at which simulation takes place
    epsilon = 0.2

    # number of frames to simulate
    frames = 5000

    # decoding iterations
    iterations = 200

    H = getRegularH(n, dv, dc)

    # simulate all-zero codeword
    x = np.zeros(n)

    errors = 0
    for frame in tqdm(range(frames)):

        if frame % 20 == 0:
            # generate parity-check matrix of regular LDPC code on a regular
            # basis so that we average over the parity-check matrices
            H = getRegularH(n, dv, dc)
        
        # erasure channel, first map to bipolar and map to very large value as
        # approximation to infinite LLR 
        y = (1 - 2*x).astype(int)
        y[np.random.rand(x.size) < epsilon] = 2 # 2 denotes erasure

        xh = decode_LDPC_BEC(y, H, iterations)

        errors += (xh.size == 0)    

        if frame % 20 == 0:
            print(f"\rSimulating ... current estimate P_E = {errors/(frame+1):.2e}", end="\r", flush=True)  

    FER = errors / frames # divide by two, as we may correctly guess the residual errors
    print(f"\nepsilon = {epsilon:.2f}: FER = {FER:.4g}") 


In [6]:
simulate_LDPC_BEC()

  0%|          | 0/5000 [00:00<?, ?it/s]

Simulating ... current estimate P_E = 2.61e-03
epsilon = 0.20: FER = 0.0026
